In [1]:
# Import required libraries
from google.cloud import bigquery
import pandas as pd

In [2]:
# Initialize BigQuery client
client = bigquery.Client()

In [3]:
# Configuration
PROJECT_ID = 'qwiklabs-gcp-04-af615cbd429c'
DATASET_ID = 'fraud_detection'
SOURCE_FILE = 'gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv'
RAW_TABLE = 'fraud_data_raw'
FINAL_TABLE = 'fraud_training_data'

In [4]:
# Create the dataset if it doesn't exist
dataset_id = f"{PROJECT_ID}.{DATASET_ID}"

try:
    client.get_dataset(dataset_id)
    print(f"Dataset {dataset_id} already exists")
except:
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = "US"
    dataset = client.create_dataset(dataset, timeout=30)
    print(f"Created dataset {dataset_id}")

Created dataset qwiklabs-gcp-04-af615cbd429c.fraud_detection


In [5]:
# Load CSV from GCS into BigQuery
load_job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition='WRITE_TRUNCATE'
)

uri = SOURCE_FILE
table_id = f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}"

load_job = client.load_table_from_uri(
    uri,
    table_id,
    job_config=load_job_config
)

load_job.result()  # Wait for the job to complete

print(f"Loaded {load_job.output_rows} rows into {table_id}")

Loaded 50000 rows into qwiklabs-gcp-04-af615cbd429c.fraud_detection.fraud_data_raw


In [6]:
# Preview the raw data
query = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}`
LIMIT 5
"""

df_preview = client.query(query).to_dataframe()
print("Raw Data Preview:")
print(df_preview)
print("\nColumn Names:", df_preview.columns.tolist())

Raw Data Preview:
   Applicant_ID  Age Employment_Status  Income  Number_of_Dependents  \
0           217   65        Unemployed   28984                     4   
1           226   54     Self-Employed       0                     1   
2           240   26     Self-Employed   64477                     5   
3           252   28        Unemployed   28576                     4   
4           266   43          Employed   44930                     5   

   Amount_Requested  Previous_Assistance_Received Previous_Assistance_Date  \
0              5872                         False                      NaT   
1              6631                         False                      NaT   
2              8612                         False                      NaT   
3              2951                         False                      NaT   
4              2324                         False                      NaT   

   Supporting_Doc_Verified  Application_Frequency_Last_Year       IP_Address  \


In [20]:
# Feature engineering query
query = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{FINAL_TABLE}` AS

WITH base_data AS (
  SELECT
    *,
    CASE
      WHEN Age BETWEEN 18 AND 24 THEN '18-24'
      WHEN Age BETWEEN 25 AND 34 THEN '25-34'
      WHEN Age BETWEEN 35 AND 44 THEN '35-44'
      WHEN Age BETWEEN 45 AND 54 THEN '45-54'
      WHEN Age BETWEEN 55 AND 64 THEN '55-64'
      WHEN Age >= 65 THEN '65+'
      ELSE 'Unknown'
    END AS Age_Bin
  FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}`
),

feature_engineering AS (
  SELECT
    * EXCEPT(Employment_Status, Device_Type, Age_Bin, Previous_Assistance_Received, Supporting_Doc_Verified),

    -- One-hot encode Employment_Status
    CASE WHEN Employment_Status = 'Employed' THEN 1 ELSE 0 END AS Employment_Status_Employed,
    CASE WHEN Employment_Status = 'Unemployed' THEN 1 ELSE 0 END AS Employment_Status_Unemployed,
    CASE WHEN Employment_Status = 'Self-Employed' THEN 1 ELSE 0 END AS Employment_Status_Self_Employed,

    -- One-hot encode Device_Type
    CASE WHEN Device_Type = 'Mobile' THEN 1 ELSE 0 END AS Device_Type_Mobile,
    CASE WHEN Device_Type = 'Desktop' THEN 1 ELSE 0 END AS Device_Type_Desktop,
    CASE WHEN Device_Type = 'Tablet' THEN 1 ELSE 0 END AS Device_Type_Tablet,

    -- One-hot encode Age Bins
    CASE WHEN Age_Bin = '18-24' THEN 1 ELSE 0 END AS Age_Bin_18_24,
    CASE WHEN Age_Bin = '25-34' THEN 1 ELSE 0 END AS Age_Bin_25_34,
    CASE WHEN Age_Bin = '35-44' THEN 1 ELSE 0 END AS Age_Bin_35_44,
    CASE WHEN Age_Bin = '45-54' THEN 1 ELSE 0 END AS Age_Bin_45_54,
    CASE WHEN Age_Bin = '55-64' THEN 1 ELSE 0 END AS Age_Bin_55_64,
    CASE WHEN Age_Bin = '65+' THEN 1 ELSE 0 END AS Age_Bin_65_Plus,

    -- Create Income-to-Amount-Requested ratio
    SAFE_DIVIDE(Income, Amount_Requested) AS Income_to_Amount_Requested,

    -- Time_Since_Previous_Assistance
    DATE_DIFF(
      CURRENT_DATE(),
      CAST(Previous_Assistance_Date AS DATE),
      DAY
    ) AS Time_Since_Previous_Assistance,

    -- Convert True/False fields to 0/1
    CASE WHEN Previous_Assistance_Received THEN 1 ELSE 0 END AS Previous_Assistance_Received_Binary,
    CASE WHEN Supporting_Doc_Verified THEN 1 ELSE 0 END AS Supporting_Doc_Verified_Binary

  FROM base_data
)

SELECT * FROM feature_engineering
"""

try:
    job = client.query(query)
    job.result()
    print(f"SUCCESS! Table '{FINAL_TABLE}' created")
except Exception as e:
    print(f"Error: {e}")

SUCCESS! Table 'fraud_training_data' created


In [21]:
verify_query = f"""
SELECT
  COUNT(*) as total_rows,
  AVG(Income_to_Amount_Requested) as avg_income_ratio,
  SUM(Employment_Status_Employed) as employed_count,
  SUM(Device_Type_Mobile) as mobile_count
FROM `{PROJECT_ID}.{DATASET_ID}.{FINAL_TABLE}`
"""

df = client.query(verify_query).to_dataframe()
print("\n Feature Engineering Results:")
print(df)


 Feature Engineering Results:
   total_rows  avg_income_ratio  employed_count  mobile_count
0       50000          6.664208           16734         16733


In [22]:
preview_query = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.{FINAL_TABLE}`
LIMIT 5
"""

preview_df = client.query(preview_query).to_dataframe()
print(f"\n Columns: {list(preview_df.columns)}")
print(f"Shape: {preview_df.shape}")
display(preview_df)


 Columns: ['Applicant_ID', 'Age', 'Income', 'Number_of_Dependents', 'Amount_Requested', 'Previous_Assistance_Date', 'Application_Frequency_Last_Year', 'IP_Address', 'Application_Date', 'Fraudulent', 'Employment_Status_Employed', 'Employment_Status_Unemployed', 'Employment_Status_Self_Employed', 'Device_Type_Mobile', 'Device_Type_Desktop', 'Device_Type_Tablet', 'Age_Bin_18_24', 'Age_Bin_25_34', 'Age_Bin_35_44', 'Age_Bin_45_54', 'Age_Bin_55_64', 'Age_Bin_65_Plus', 'Income_to_Amount_Requested', 'Time_Since_Previous_Assistance', 'Previous_Assistance_Received_Binary', 'Supporting_Doc_Verified_Binary']
Shape: (5, 26)


,Applicant_ID,Age,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Date,Application_Frequency_Last_Year,IP_Address,Application_Date,Fraudulent,...,Age_Bin_18_24,Age_Bin_25_34,Age_Bin_35_44,Age_Bin_45_54,Age_Bin_55_64,Age_Bin_65_Plus,Income_to_Amount_Requested,Time_Since_Previous_Assistance,Previous_Assistance_Received_Binary,Supporting_Doc_Verified_Binary
0,553,27,0,3,3104,NaT,1,202.18.34.83,2024-11-04,0,...,0,1,0,0,0,0,0.0,<NA>,0,0
1,1033,62,0,3,9950,NaT,1,129.60.142.205,2024-04-27,0,...,0,0,0,0,1,0,0.0,<NA>,0,0
2,2879,27,0,0,4580,NaT,1,83.40.216.209,2024-01-02,0,...,0,1,0,0,0,0,0.0,<NA>,0,1
3,2890,44,0,3,2251,NaT,1,66.204.127.219,2024-04-06,0,...,0,0,1,0,0,0,0.0,<NA>,0,0
4,3504,24,0,0,2346,NaT,1,66.204.98.138,2024-01-22,0,...,1,0,0,0,0,0,0.0,<NA>,0,0
